In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import krippendorff
import seaborn as sns
from itertools import combinations

columns_of_interest = ['FOOD', 'DESCR', 'CURE']
filepaths = ['cure_1_bio.tsv', 'cure_2_bio.tsv', 'LLM_Cure_nested.tsv']

annotator_names = ['Annot1', 'Annot2', 'LLM']

def load_and_clean(filepath):
    df = pd.read_csv(filepath, sep="\t", dtype=str).fillna("O")
    df = df[columns_of_interest]
    df = df.applymap(lambda x: re.sub(r'\\_', '_', x))
    return df

dfs = [load_and_clean(fp) for fp in filepaths]
human_dfs = dfs[:2]
llm_df = dfs[2]

# for i, df in enumerate(human_dfs + [llm_df]):
#     if len(df) != len(llm_df):
#         raise ValueError(f"Mismatch in length between LLM and {annotator_names[i]}: {len(df)} vs {len(llm_df)}")


for idx, human_df in enumerate(human_dfs):
    print(f"\nAgreement {annotator_names[idx]} amd LLM")
    for col in columns_of_interest:
        human_col = human_df[col].tolist()
        llm_col = llm_df[col].tolist()

        labels = sorted(set(human_col + llm_col), key=lambda x: (x != 'O', x))
        
        alpha = krippendorff.alpha(reliability_data=np.array([human_col, llm_col]), level_of_measurement='nominal')
        print(f"{col} - Krippendorff's alpha: {alpha:.4f}")

        cm = confusion_matrix(human_col, llm_col, labels=labels)
        disagreements = cm.sum() - np.trace(cm)
        percent_disagreement = disagreements / cm.sum() * 100

        fig, ax = plt.subplots(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
        ax.set_title(f"{col}: {annotator_names[idx]} vs LLM", fontsize=10)
        ax.set_xlabel("LLM")
        ax.set_ylabel(annotator_names[idx])
        plt.suptitle(f"Disagreement: {disagreements} ({percent_disagreement:.2f}%)", fontsize=10, y=0.94)
        plt.tight_layout()
        plt.show()

print(f"\nAgreement {annotator_names[0]} e {annotator_names[1]}")
for col in columns_of_interest:
    col1 = human_dfs[0][col].tolist()
    col2 = human_dfs[1][col].tolist()

    labels = sorted(set(col1 + col2), key=lambda x: (x != 'O', x))

    alpha = krippendorff.alpha(reliability_data=np.array([col1, col2]), level_of_measurement='nominal')
    print(f"{col} - Krippendorff's alpha: {alpha:.4f}")

    cm = confusion_matrix(col1, col2, labels=labels)
    disagreements = cm.sum() - np.trace(cm)
    percent_disagreement = disagreements / cm.sum() * 100

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
    ax.set_title(f"{col}: {annotator_names[0]} vs {annotator_names[1]}", fontsize=10)
    ax.set_xlabel(annotator_names[1])
    ax.set_ylabel(annotator_names[0])
    plt.suptitle(f"Disagreement: {disagreements} ({percent_disagreement:.2f}%)", fontsize=10, y=0.94)
    plt.tight_layout()
    plt.show()

concatenated_annotations = []
for df in dfs:
    concatenated = []
    for col in columns_of_interest:
        concatenated.extend(df[col].tolist())
    concatenated_annotations.append(concatenated)


print("\n" + "="*50)
print("KRIPPENDORFF'S ALPHA TRA ANNOTATORI (GLOBALE E COPPIE)")
print("="*50)

data_matrix = np.array(concatenated_annotations)
alpha_global = krippendorff.alpha(reliability_data=data_matrix, level_of_measurement='nominal')
print(f"\nKrippendorff's alpha (Annot1 vs Annot2 vs LLM): {alpha_global:.4f}")

annotator_names_full = ['Annot1', 'Annot2', 'LLM']
pairs = [(0, 1), (0, 2), (1, 2)]

for i, j in pairs:
    pair_matrix = np.array([concatenated_annotations[i], concatenated_annotations[j]])
    alpha_pair = krippendorff.alpha(reliability_data=pair_matrix, level_of_measurement='nominal')
    print(f"Krippendorff's alpha ({annotator_names_full[i]} vs {annotator_names_full[j]}): {alpha_pair:.4f}")

print("\n" + "="*50)
print("KRIPPENDORFF'S ALPHA PER COLONNA (Annot1 vs Annot2 vs LLM)")
print("="*50)

for col in columns_of_interest:
    data_matrix = np.array([
        dfs[0][col].tolist(),
        dfs[1][col].tolist(),
        dfs[2][col].tolist()
    ])
    alpha = krippendorff.alpha(reliability_data=data_matrix, level_of_measurement='nominal')
    print(f"{col}: Krippendorff's α = {alpha:.4f}")